In [1]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np
import faiss

In [2]:
# Bi-Encoder
bi_encoder = SentenceTransformer('Qwen/Qwen3-Embedding-0.6B')

In [3]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [4]:
documents = [
    "LangChain helps build LLM applications.",
    "FAISS is used for vector similarity search.",
    "Cross encoders are more accurate but slower.",
    "Bi-encoders are fast but less accurate.",
    "RAG combines retrieval with generation.",
    "Transformers power modern NLP systems.",
    "Embeddings convert text to vectors.",
    "Vector databases store embeddings.",
    "OpenAI provides powerful LLM APIs.",
    "Re-ranking improves search quality."
]

In [5]:
# bi encoder steps - encode both query and documents
docs_embeddings = bi_encoder.encode(documents, convert_to_numpy=True)

In [6]:
docs_embeddings

array([[ 0.02401425,  0.05502725, -0.00689245, ...,  0.01757455,
         0.00412146, -0.01295322],
       [-0.02135323, -0.0250708 , -0.00864905, ..., -0.0141419 ,
         0.02171834, -0.00324752],
       [ 0.00292395,  0.0011832 , -0.00938225, ..., -0.03133884,
         0.0085481 , -0.03422589],
       ...,
       [-0.04540006, -0.02645862, -0.0090127 , ..., -0.05655018,
         0.03737333,  0.02883572],
       [ 0.00440428,  0.03201355, -0.00915708, ...,  0.02214475,
         0.00621312,  0.00273107],
       [-0.0333166 , -0.03485272, -0.0108177 , ..., -0.00694886,
         0.02665143, -0.01973763]], shape=(10, 1024), dtype=float32)

In [7]:
docs_embeddings.shape[1]

1024

In [8]:
dimensions = docs_embeddings.shape[1]

# index for FAISS
index = faiss.IndexFlatL2(dimensions)
index.add(docs_embeddings)

In [9]:
# user Query
user_query = "How to improve retrieval accuracy?"

In [10]:
query_embedding = bi_encoder.encode([user_query])

In [11]:
top_k = 10
distance, indices = index.search(query_embedding, top_k)

print(f"distance : {distance}, indices : {indices}")

distance : [[0.50092494 0.5602648  0.72863746 0.75214434 0.7576274  0.8610383
  0.8678126  0.8814485  0.9674803  1.0264437 ]], indices : [[9 4 1 5 8 2 0 3 7 6]]


In [12]:
top_chunks = [documents[i] for i in indices[0]]
top_chunks

['Re-ranking improves search quality.',
 'RAG combines retrieval with generation.',
 'FAISS is used for vector similarity search.',
 'Transformers power modern NLP systems.',
 'OpenAI provides powerful LLM APIs.',
 'Cross encoders are more accurate but slower.',
 'LangChain helps build LLM applications.',
 'Bi-encoders are fast but less accurate.',
 'Vector databases store embeddings.',
 'Embeddings convert text to vectors.']

In [13]:
# Re-Rank using Cross Encoder
from sentence_transformers import CrossEncoder

# create pair with query i.e [ (query, chunk) ... ]
query_chunk_pairs = [(user_query, chunk) for chunk in top_chunks]

query_chunk_pairs

[('How to improve retrieval accuracy?', 'Re-ranking improves search quality.'),
 ('How to improve retrieval accuracy?',
  'RAG combines retrieval with generation.'),
 ('How to improve retrieval accuracy?',
  'FAISS is used for vector similarity search.'),
 ('How to improve retrieval accuracy?',
  'Transformers power modern NLP systems.'),
 ('How to improve retrieval accuracy?', 'OpenAI provides powerful LLM APIs.'),
 ('How to improve retrieval accuracy?',
  'Cross encoders are more accurate but slower.'),
 ('How to improve retrieval accuracy?',
  'LangChain helps build LLM applications.'),
 ('How to improve retrieval accuracy?',
  'Bi-encoders are fast but less accurate.'),
 ('How to improve retrieval accuracy?', 'Vector databases store embeddings.'),
 ('How to improve retrieval accuracy?', 'Embeddings convert text to vectors.')]

In [14]:
# get scores for each pairs
scores = cross_encoder.predict(query_chunk_pairs)

scores

array([ -7.021578,  -6.903639, -10.939784, -11.347504, -11.312512,
        -9.876415, -11.283894,  -9.824383, -11.305241, -11.248499],
      dtype=float32)

In [15]:
# Re-Rank by score
reranked_chunks = sorted(zip(top_chunks, scores), key=lambda x:x[1], reverse=True)
reranked_chunks

[('RAG combines retrieval with generation.', np.float32(-6.903639)),
 ('Re-ranking improves search quality.', np.float32(-7.021578)),
 ('Bi-encoders are fast but less accurate.', np.float32(-9.824383)),
 ('Cross encoders are more accurate but slower.', np.float32(-9.876415)),
 ('FAISS is used for vector similarity search.', np.float32(-10.939784)),
 ('Embeddings convert text to vectors.', np.float32(-11.248499)),
 ('LangChain helps build LLM applications.', np.float32(-11.283894)),
 ('Vector databases store embeddings.', np.float32(-11.305241)),
 ('OpenAI provides powerful LLM APIs.', np.float32(-11.312512)),
 ('Transformers power modern NLP systems.', np.float32(-11.347504))]

In [16]:
# Final Top Results
for chunk, score in reranked_chunks:
    print(f"{score:.4f} → {chunk}")

-6.9036 → RAG combines retrieval with generation.
-7.0216 → Re-ranking improves search quality.
-9.8244 → Bi-encoders are fast but less accurate.
-9.8764 → Cross encoders are more accurate but slower.
-10.9398 → FAISS is used for vector similarity search.
-11.2485 → Embeddings convert text to vectors.
-11.2839 → LangChain helps build LLM applications.
-11.3052 → Vector databases store embeddings.
-11.3125 → OpenAI provides powerful LLM APIs.
-11.3475 → Transformers power modern NLP systems.


In [17]:
"""
User Query
   ↓
Bi-Encoder → Vector Search (FAISS)
   ↓
Top 10 Chunks (fast but noisy)
   ↓
Cross-Encoder → Relevance Scoring
   ↓
Re-ranked Results (high quality)
"""

'\nUser Query\n   ↓\nBi-Encoder → Vector Search (FAISS)\n   ↓\nTop 10 Chunks (fast but noisy)\n   ↓\nCross-Encoder → Relevance Scoring\n   ↓\nRe-ranked Results (high quality)\n'